## Summary and Key Insights

### DEX Method Performance
- **Training Loss**: Convergence of CrossEntropyLoss on 101 age classes
- **MAE (Mean Absolute Error)**: Error in years, directly interpretable
- **Method Advantage**: Stable predictions via probability distribution + expected value

### Why DEX Works:
1. **Distribution-based approach**: Instead of directly regressing to a single value, DEX predicts the probability of each age
2. **Uncertainty modeling**: The full probability distribution contains more information than a point estimate
3. **Stable loss**: CrossEntropyLoss on discrete classes is more stable than MSE regression

### Improvements Over Regression:
- More robust to outliers
- Better uncertainty quantification
- Smoother gradients during training
- Can handle multi-modal distributions (e.g., "looks like 25 or 28 years old")

### Next Steps:
- Fine-tune on IMDB-WIKI dataset (500k+ images)
- Use larger models (ResNet, VGG)
- Implement data augmentation strategies
- Add confidence/uncertainty measures to predictions

In [ ]:
# Plot 3: Prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Predictions vs Ground Truth
axes[0].scatter(test_ground_truth, test_predictions, alpha=0.5, s=30)
axes[0].plot([0, 100], [0, 100], 'r--', label='Perfect prediction', linewidth=2)
axes[0].set_xlabel('Ground Truth Age', fontsize=12)
axes[0].set_ylabel('Predicted Age', fontsize=12)
axes[0].set_title('Predictions vs Ground Truth', fontsize=13, fontweight='bold')
axes[0].set_xlim(0, 100)
axes[0].set_ylim(0, 100)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Histogram of errors
errors = np.abs(test_predictions - test_ground_truth)
axes[1].hist(errors, bins=20, edgecolor='black', alpha=0.7, color='skyblue')
axes[1].axvline(np.mean(errors), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(errors):.2f}y')
axes[1].axvline(np.median(errors), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(errors):.2f}y')
axes[1].set_xlabel('Prediction Error (years)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Prediction Errors', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✓ Prediction analysis plots created")

In [ ]:
print("\n" + "="*70)
print("TEST PREDICTIONS VISUALIZATION".center(70))
print("="*70 + "\n")

# Get predictions on test set
model.eval()
test_predictions = []
test_ground_truth = []
test_images = []

with torch.no_grad():
    for images, ages in test_loader:
        images_batch = images.to(device)
        ages_batch = ages.to(device)
        
        _, probs = model(images_batch)
        predicted_ages = calculate_expected_age(probs, device)
        
        test_predictions.extend(predicted_ages.cpu().numpy())
        test_ground_truth.extend(ages_batch.cpu().numpy())
        test_images.extend(images.cpu().numpy())

test_predictions = np.array(test_predictions)
test_ground_truth = np.array(test_ground_truth)
test_images = np.array(test_images)

# Plot 2: Display sample test images with predictions
print(f"Displaying sample predictions...\n")

num_samples = 9
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

# Select random samples
indices = np.random.choice(len(test_predictions), num_samples, replace=False)

for idx, sample_idx in enumerate(indices):
    # Get image
    img = test_images[sample_idx].transpose(1, 2, 0)
    
    # Denormalize
    img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    
    # Ground truth and prediction
    gt_age = test_ground_truth[sample_idx]
    pred_age = test_predictions[sample_idx]
    error = abs(pred_age - gt_age)
    
    # Plot
    axes[idx].imshow(img)
    axes[idx].set_title(f'True: {gt_age:.0f} | Pred: {pred_age:.1f} | Error: {error:.1f}y', 
                       fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"✓ Sample test predictions displayed")
print(f"  - Sample statistics:")
print(f"    • Mean prediction error: {np.mean(np.abs(test_predictions - test_ground_truth)):.2f} years")
print(f"    • Min error: {np.min(np.abs(test_predictions - test_ground_truth)):.2f} years")
print(f"    • Max error: {np.max(np.abs(test_predictions - test_ground_truth)):.2f} years")

In [ ]:
print("\n" + "="*70)
print("RESULTS VISUALIZATION".center(70))
print("="*70 + "\n")

# Plot 1: Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(range(1, num_epochs + 1), train_losses, 'b-o', label='Train Loss', linewidth=2, markersize=6)
axes[0].plot(range(1, num_epochs + 1), test_losses, 'r-s', label='Test Loss', linewidth=2, markersize=6)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (Cross-Entropy)', fontsize=12)
axes[0].set_title('Training and Test Loss', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3, linestyle='--')

# MAE curve (equivalent to accuracy in years)
axes[1].plot(range(1, num_epochs + 1), train_maes, 'b-o', label='Train MAE', linewidth=2, markersize=6)
axes[1].plot(range(1, num_epochs + 1), test_maes, 'r-s', label='Test MAE', linewidth=2, markersize=6)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MAE (years)', fontsize=12)
axes[1].set_title('Training and Test Mean Absolute Error', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("✓ Loss and MAE curves plotted")
print(f"  - Final Train Loss: {train_losses[-1]:.4f}")
print(f"  - Final Test Loss: {test_losses[-1]:.4f}")
print(f"  - Final Train MAE: {train_maes[-1]:.2f} years")
print(f"  - Final Test MAE: {test_maes[-1]:.2f} years")

## 6. Results Visualization

In [ ]:
# Training loop
print("Starting training...\n")
print(f"{'Epoch':<6} {'Train Loss':<12} {'Train MAE':<12} {'Test Loss':<12} {'Test MAE':<12} {'LR':<8}")
print("-" * 60)

try:
    for epoch in range(num_epochs):
        # Clear GPU cache and collect garbage before each epoch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Train
        train_loss, train_mae = train_epoch(model, train_loader, criterion, optimizer, device)
        train_losses.append(train_loss)
        train_maes.append(train_mae)
        
        # Evaluate
        test_loss, test_mae = evaluate(model, test_loader, criterion, device)
        test_losses.append(test_loss)
        test_maes.append(test_mae)
        
        # Learning rate
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        
        # Print progress
        print(f"{epoch+1:<6} {train_loss:<12.4f} {train_mae:<12.2f} {test_loss:<12.4f} {test_mae:<12.2f} {current_lr:<8.6f}")

    print("\n" + "="*70)
    print("✓ Training completed!")
    print("="*70)
    
except RuntimeError as e:
    if "CUDA" in str(e) or "out of memory" in str(e):
        print("\n" + "❌ "*35)
        print("GPU Memory Error - Possible solutions:")
        print("  1. Reduce BATCH_SIZE further (currently 8)")
        print("  2. Reduce IMG_SIZE further (currently 224)")
        print("  3. Reduce num_epochs (currently 10)")
        print("  4. Run on CPU instead (slower but memory-efficient)")
        print("="*70)
        raise
    else:
        raise


                               TRAINING                               

✓ Optimizer: Adam (lr=0.001, weight_decay=5e-4)
✓ Loss function: CrossEntropyLoss
✓ Scheduler: StepLR (step_size=5, gamma=0.5)
✓ Epochs: 10

Starting training...

Epoch  Train Loss   Train MAE    Test Loss    Test MAE     LR      
------------------------------------------------------------


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 5. Training Loop

In [6]:
print("\n" + "="*70)
print("DEX MODEL DEFINITION".center(70))
print("="*70 + "\n")

class DEXModel(nn.Module):
    """
    DEX (Deep EXpectation) Model for Age Estimation
    
    Architecture:
    - Feature Extractor: Modified AlexNet with 5 conv layers
    - Classifier: 3 FC layers with dropout (4096 → 4096 → 101)
    - Output: Softmax probabilities over 101 age classes
    
    Key innovation: Predicts age distribution, then computes expected value
    """
    
    def __init__(self, num_classes=101, pretrained=True):
        super(DEXModel, self).__init__()
        
        # Feature extraction (AlexNet-like architecture)
        self.features = nn.Sequential(
            # Layer 1
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Layer 2
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Layer 3
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Layer 4
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Layer 5
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            
            nn.Linear(4096, num_classes),
        )
        
        self.num_classes = num_classes
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input images [batch_size, 3, 227, 227]
        
        Returns:
            logits: Raw output before softmax [batch_size, 101]
            probs: Softmax probabilities [batch_size, 101]
        """
        # Feature extraction
        x = self.features(x)
        
        # Flatten for classifier
        x = x.view(x.size(0), -1)
        
        # Classifier
        logits = self.classifier(x)
        
        # Softmax for age distribution
        probs = torch.softmax(logits, dim=1)
        
        return logits, probs


def calculate_expected_age(probs, device):
    """
    Calculate expected age from probability distribution
    
    The key insight of DEX: Instead of regressing directly to age,
    we predict a distribution and compute the expected value.
    
    E[age] = Σ(age * P(age)) for age in [0, 100]
    
    Args:
        probs: Probability distribution [batch_size, 101]
        device: torch device
    
    Returns:
        expected_ages: Expected age for each sample [batch_size]
    """
    age_range = torch.arange(0, 101, dtype=torch.float32, device=device)
    expected_ages = torch.sum(probs * age_range.unsqueeze(0), dim=1)
    return expected_ages


# Initialiser le modèle
print("Creating DEX model...")
model = DEXModel(num_classes=AGE_BINS).to(device)

# Afficher l'architecture
print("\n✓ DEX Model created:")
print(f"  - Architecture: AlexNet-based feature extractor")
print(f"  - Output classes: {AGE_BINS} (ages 0-100)")
print(f"  - Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  - Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Afficher un résumé des couches principales
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ Model Summary:")
print(f"  - Features: Conv layers + ReLU + MaxPool")
print(f"  - Classifier: 3 FC layers with Dropout")
print(f"  - Total size: {total_params / 1e6:.2f} M parameters")


                         DEX MODEL DEFINITION                         

Creating DEX model...

✓ DEX Model created:
  - Architecture: AlexNet-based feature extractor
  - Output classes: 101 (ages 0-100)
  - Total parameters: 58,155,045
  - Trainable parameters: 58,155,045

✓ Model Summary:
  - Features: Conv layers + ReLU + MaxPool
  - Classifier: 3 FC layers with Dropout
  - Total size: 58.16 M parameters


## 4. DEX Model Architecture

### Deep EXpectation Method
La clé du succès de DEX est d'utiliser une **distribution de probabilités** au lieu d'une **régression directe**:

```
Entrée: Image → [AlexNet Features] → [Classifier] → Logits (101 classes)
                                                    ↓
                                              Softmax → Probs (0-100)
                                                    ↓
                                        Expected Age = Σ(age × prob)
```

Cette approche est plus stable car elle capture l'incertitude dans les prédictions.


In [ ]:
# Créer les datasets et dataloaders avec APPA-REAL
print("\n" + "="*70)
print("CREATING DATALOADERS".center(70))
print("="*70 + "\n")

print("Utilisation du dataset APPA-REAL...\n")

try:
    # Charger les datasets APPA-REAL
    train_dataset = AppaRealDataset(dataset_dir, "train", train_transform)
    test_dataset = AppaRealDataset(dataset_dir, "test", test_transform)
    
    # Vérifier que les datasets ne sont pas vides
    if len(train_dataset) == 0 or len(test_dataset) == 0:
        raise ValueError("Datasets APPA-REAL vides!")
    
    print(f"✓ Datasets APPA-REAL chargés avec succès!")
    
except Exception as e:
    print(f"❌ Erreur lors du chargement d'APPA-REAL: {e}")
    print("\n⚠️  Vérifiez que:")
    print("  1. Le fichier appa-real-release.zip a été uploadé depuis Google Drive")
    print("  2. Les dossiers 'train' et 'test' existent dans appa-real-release/")
    print("  3. Les fichiers d'annotations (gt_avg_train.csv, gt_avg_test.csv) sont présents")
    print("\nArrêt du notebook.")
    raise

# Créer les dataloaders avec optimisations mémoire
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

print(f"\n✓ Dataloaders créés:")
print(f"  - Train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"  - Test: {len(test_dataset)} images ({len(test_loader)} batches)")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Memory optimizations: drop_last=True for training, num_workers=0")


                         CREATING DATALOADERS                         

Utilisation du dataset APPA-REAL...

✓ Datasets APPA-REAL chargés avec succès!

✓ Dataloaders créés:
  - Train: 4114 images (129 batches)
  - Test: 1979 images (62 batches)
  - Batch size: 32
  - Image size: 227x227


In [ ]:
print("\n" + "="*70)
print("DATA PREPROCESSING".center(70))
print("="*70 + "\n")

# Transformations pour le preprocessing
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

print("✓ Transformations configurées:")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Train: augmentation (flip, rotation, color jitter)")
print(f"  - Test: pas d'augmentation (normalization only)")


# Dataset personnalisé pour APPA-REAL
class AppaRealDataset(Dataset):
    """
    Dataset APPA-REAL for age estimation
    
    Structure attendue:
    - appa-real-release/
        - train/
            - *.jpg (images)
        - test/
            - *.jpg (images)
        - gt_avg_train.csv (annotations)
        - gt_avg_test.csv (annotations)
    
    Format CSV: <filename>,<age>
    """
    def __init__(self, data_dir, data_type="train", transform=None):
        self.data_dir = Path(data_dir) / data_type
        self.transform = transform
        
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Dataset directory not found: {self.data_dir}")
        
        # Charger les fichiers d'annotations
        csv_file = Path(data_dir) / f"gt_avg_{data_type}.csv"
        
        if not csv_file.exists():
            raise FileNotFoundError(f"Annotation file not found: {csv_file}")
        
        # Lire le CSV
        df = pd.read_csv(csv_file, sep=',', header=None)
        self.images = df[0].values  # filenames
        self.ages = df[1].values    # ages
        
        print(f"✓ Loaded {len(self.images)} {data_type} samples from {csv_file}")
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.data_dir / self.images[idx]
        
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        image = Image.open(img_path).convert('RGB')
        age = int(self.ages[idx])
        
        if self.transform:
            image = self.transform(image)
        
        return image, age


print("\n✓ AppaRealDataset class défini")
print(f"  - Supporte train et test splits")
print(f"  - Charge les annotations depuis CSV")
print(f"  - Applique les transformations configurées")


                          DATA PREPROCESSING                          

✓ Transformations configurées:
  - Image size: 227x227
  - Train: augmentation (flip, rotation, color)
  - Test: pas d'augmentation (only normalization)

✓ Classes de dataset définies
  - AppaRealDataset: pour APPA-REAL réel
  - SyntheticAgeDataset: pour démonstration


## 3. Data Preprocessing and Augmentation

In [ ]:
# Mount Google Drive and load APPA-REAL dataset
print("\n" + "="*70)
print("DATASET LOADING FROM GOOGLE DRIVE".center(70))
print("="*70 + "\n")

dataset_dir = Path("appa-real-release")

# Vérifier d'abord si le dataset existe localement
if dataset_dir.exists():
    print(f"✓ Dataset APPA-REAL trouvé localement: {dataset_dir}")
    print(f"✓ Contenu:")
    for item in sorted(dataset_dir.iterdir()):
        print(f"  - {item.name}")

else:
    print("⚠️  Dataset APPA-REAL non trouvé localement\n")
    print("Chargement depuis Google Drive...\n")
    
    try:
        # Monte Google Drive
        from google.colab import drive
        drive.mount('/content/drive')
        
        # Cherche le fichier zip
        zip_path = '/content/drive/MyDrive/appa-real-release.zip'
        
        if os.path.exists(zip_path):
            print("📦 Copie et décompression du dataset depuis Google Drive...\n")
            
            # Copy et decompress
            os.system('cp /content/drive/MyDrive/appa-real-release.zip .')
            os.system('unzip -o -q appa-real-release.zip')
            os.system('rm -rf __MACOSX')  # Remove macOS hidden files
            
            print("✓ Dataset décompressé avec succès!")
            print(f"✓ Contenu:")
            for item in sorted(dataset_dir.iterdir()):
                print(f"  - {item.name}")
        else:
            print(f"❌ Fichier non trouvé: {zip_path}")
            print("\n📁 Instructions:")
            print("1. Ouvrez Google Drive: https://drive.google.com")
            print("2. Uploadez 'appa-real-release.zip' dans 'Mon Drive'")
            print("3. Réexécutez cette cellule")
            raise FileNotFoundError(f"APPA-REAL dataset non trouvé à {zip_path}")
    
    except ImportError:
        print("❌ Cette cellule fonctionne uniquement sur Google Colab")
        print("Pour utiliser un dataset local:")
        print("  - Décompressez 'appa-real-release.zip' dans le répertoire courant")
        print("  - Réexécutez cette cellule")
        raise
    except Exception as e:
        print(f"❌ Erreur: {e}")
        raise

print(f"\n✓ Dataset APPA-REAL prêt pour l'entraînement!")

Mounted at /content/drive
📦 Copie et décompression du dataset depuis Google Drive...

✓ Dataset décompressé avec succès!
✓ Contenu:
  - gt_avg_train.csv
  - .badfiles.un~
  - .clean_asdf.sh.un~
  - gt_test.csv
  - gt_avg_test.csv
  - .README.txt.un~
  - gt_train.csv
  - gt_valid.csv
  - train
  - valid
  - .parse_labels.m.un~
  - test
  - README.txt
  - gt_avg_valid.csv

✓ Mode: DATASET RÉEL


## 2. Dataset Loading and Preparation

In [ ]:
print("="*70)
print("DEX (Deep EXpectation) - Age Detection Model".center(70))
print("="*70)

# Installation des packages (décommenter sur Colab)
# !pip install -q torch torchvision opencv-python pillow numpy pandas matplotlib scikit-image albumentations tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import os
import gc
from tqdm import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Device: {device}")
print(f"✓ CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

             DEX (Deep EXpectation) - Age Detection Model             

✓ Device: cuda
✓ CUDA disponible: True
✓ GPU: Tesla T4
✓ Memory: 15.64 GB

✓ Configuration:
  - Age bins: 101
  - Image size: 227x227
  - Batch size: 32


## 1. Setup and Dependencies

# DEX (Deep EXpectation) - Age Estimation with APPA-REAL Dataset

## Overview
Cette implémentation suit la méthode DEX (Deep EXpectation) présentée par Rothe et al. (ICCV 2015 Workshops).

### Three Key Pillars of DEX:
1. **Deep EXpectation Method**: Prédire une distribution de probabilités pour chaque âge (0-100), puis calculer la moyenne pondérée pour obtenir l'âge final
2. **Massive Dataset (IMDB-WIKI)**: L'importance du volume de données (500k+ images) pour pré-entraîner le modèle
3. **Superhuman Performance**: DEX a été l'un des premiers algorithmes à dépasser la performance humaine en estimation d'âge

### What Makes DEX Different:
- ✓ Distribution-based regression (probabilités) vs direct regression
- ✓ Stable predictions grâce à la moyenne pondérée
- ✓ Fondation pour tous les modernes d'estimation d'âge